<div class='alert alert-warning'>

SciPy's interactive examples with Jupyterlite are experimental and may not always work as expected. Execution of cells containing imports may result in large downloads (up to 60MB of content for the first import from SciPy). Load times when importing from SciPy may take roughly 10-20 seconds. If you notice any problems, feel free to open an [issue](https://github.com/scipy/scipy/issues/new/choose).

</div>

The following example compares the magnitude spectra of a periodic 10-point Hann
window to its continuous-time counterpart.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal.windows import hann

M, T = 10, 1/10  # number of samples and sampling interval in seconds
k, tau = np.arange(M) * T, M*T  # sample times and window width in seconds

w_k = hann(M, sym=False)  # periodic hann window
W, f_W = fft(w_k) / sum(w_k), fftfreq(M, T)  # amplitude spectrum of w_k

# Hann function and its Fourier transform:
t = np.linspace(-.25, M*T+.25, 200, endpoint=True)
h = np.where(np.logical_and(0 < t, t < 1), np.sin(np.pi * t / tau) ** 2, 0)

f = np.linspace(-6, 12., 1801, endpoint=True)
H, ii0 = np.zeros_like(f), abs(f * tau) != 1
H[ii0] = np.sinc(f[ii0] * tau) / ((f[ii0] * tau + 1) * (f[ii0] * tau - 1))
H[~ii0] = 0.5  # value when denominator is zero

ii1 = np.logical_and(min(f_W) - 0.5 <= f, f < max(f_W) + 0.5)
H_abs, f_abs, H1, f_dB  = abs(H[ii1]), f[ii1], abs(H[f>=0]), f[f>=0]
H_dB = np.where(~np.logical_and(f_dB>=2, np.mod(f_dB, 1)==0),
                20*np.log10(H1), -1e250)

ax0, ax1, ax2 = (plt.subplots(num=n_, constrained_layout=True)[1]
                  for n_ in range(3))
ax0.set_title(r"Periodic Hann window and Hann function of width $\tau=1\,$s")
ax0.set(ylabel="Amplitude", xlim=(t[0], t[-1]),
        xlabel=rf"Time $t$ in seconds (${M}$ samples with interval ${T=}\,$s)")
ax0.plot(t, h, 'C0-', label=r"$\sin^2(\pi t / \tau)$")
ax0.plot(k, w_k, 'C1o', label="Window $w_p[kT]$")
ax1.set_title(r"Magnitude Spectrum of Hann window and Hann function")
ax1.set(ylabel="Magnitude", xlim=(f_abs[0], f_abs[-1]),
        xlabel=rf"Frequency $f$ in hertz ($\Delta f = {f_W[1]:g}\,$Hz)")
ax1.plot(f_abs, H_abs, 'C0-', label="$|H(f)|$")
ax1.plot(f_W, abs(W), 'C1o', label=r"$|W_p[l\Delta f]|$")
ax2.set_title(r"Magnitude Spectrum of Hann function")
ax2.set(ylabel=r"Magnitude $20\,\log_{10} |H(l)|$ in dB",
        xlabel=r"Relative Frequency $l=f/\Delta f\ $ ($\Delta f = 1/\tau$)",
        xlim=(f_dB[0], f_dB[-1]), ylim=(-80, 1),
        yticks=20*np.arange(-4, 1), xticks=np.arange(12))
ax2a = ax2.twinx() # create right y-axis with non-logarithmic scaling:
ax2a.set_ylabel("Magnitude $|H(f)|$", rotation=-90, labelpad=15)
ax2a.set(ylim=(1e-4, 10 ** (1 / 20)), yscale="log")
ax2.plot(f_dB, H_dB, 'C0-', label=r"$|H(l\Delta f)|$")
ax2.plot(f_dB[f_dB>0.5], -60*np.log10(f_dB[f_dB>0.5]) - 20*np.log10(np.pi),
         'C2--', alpha=0.5, label=r"$\pi^{-1} (l\Delta f)^{-3}$")
for ax_ in (ax0, ax1, ax2):
    ax_.legend(loc='best')
    ax_.grid(True)
plt.show()

The plot of the logarithmically scaled magnitude spectrum illustrates that its
zeros are at integers with absolute value ≥ 2 and that the sidelobes decrease on
the order of $O(|f|^{-3})$, which corresponds to -60 dB per frequency decade.
